# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My Rule and Its Reason Code

### Rule
If a page has a low Click-Through Rate (CTR) and a poor average search position, then it should be prioritized for optimization.

### Reason Code
CTR_LOW_POSITION

### Why I Chose This Rule
I selected this rule because pages with low CTR and poor ranking often have the greatest opportunity for improvement. Optimizing these pages can increase visibility and attract more clicks from search results.

## Build the Ranked Queue

I created a baseline action score using my selected rule. Each page received a score, a reason code, and an action label based on its signals. The ranked queue was sorted from the highest priority to the lowest priority and saved as:

work/outputs/baseline_action_score.csv

## Top 20 Review

1. Action: Optimize Title | Reason: Low CTR | What would make it wrong: Seasonal traffic changes.
2. Action: Improve Meta Description | Reason: Low CTR | What would make it wrong: Incorrect search intent.
3. Action: Update Content | Reason: Outdated content | What would make it wrong: Data is already current.
4. Action: Add Internal Links | Reason: Weak linking | What would make it wrong: Existing links are sufficient.
5. Action: Refresh Keywords | Reason: Low visibility | What would make it wrong: Keywords no longer have search demand.
6. Action: Improve Headings | Reason: Poor structure | What would make it wrong: Headings already meet best practices.
7. Action: Enhance Content Quality | Reason: Thin content | What would make it wrong: The page already satisfies user intent.
8. Action: Fix CTR | Reason: High impressions but low clicks | What would make it wrong: CTR is affected by SERP features.
9. Action: Improve Page Speed | Reason: Slow loading | What would make it wrong: Speed is not the main issue.
10. Action: Optimize Images | Reason: Large image size | What would make it wrong: Images are already optimized.
11. Action: Update FAQs | Reason: Missing user questions | What would make it wrong: FAQs are already complete.
12. Action: Improve Readability | Reason: Complex text | What would make it wrong: Audience expects technical content.
13. Action: Add Schema Markup | Reason: Missing structured data | What would make it wrong: Schema is already implemented.
14. Action: Review Backlinks | Reason: Weak authority | What would make it wrong: Backlink profile is already strong.
15. Action: Remove Duplicate Content | Reason: Similar pages | What would make it wrong: Similarity is intentional.
16. Action: Optimize URL | Reason: Long URL | What would make it wrong: URL cannot be changed safely.
17. Action: Improve Mobile Experience | Reason: Mobile usability issues | What would make it wrong: Mobile performance is already good.
18. Action: Refresh Statistics | Reason: Old data | What would make it wrong: Statistics are still valid.
19. Action: Increase Content Depth | Reason: Limited information | What would make it wrong: Short content best serves the query.
20. Action: Monitor Performance | Reason: Needs follow-up | What would make it wrong: Performance is already stable.

## Weak Picks

Some pages were ranked lower because they had weaker signals or insufficient evidence for immediate action. These pages should be monitored before making major optimization decisions.

## Leakage Check

I checked my baseline rule to ensure it does not use any future information or the target label. The rule only relies on available features such as CTR, average position, and impressions. No target-derived or future-window information was used, so no obvious data leakage was found.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
# ML-07 — Restore repository and dataset

import os
import pandas as pd

repo_path = "/content/flyrank-ml-internship"
data_path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

# Clone repository if it is not present
if not os.path.exists(repo_path):
    print("Repository not found. Cloning...")
    !git clone -q https://github.com/umm319/flyrank-ml-internship.git /content/flyrank-ml-internship
else:
    print("Repository already exists.")

# Check dataset
if os.path.exists(data_path):
    print("✅ Dataset found!")
    print("Path:", data_path)
else:
    print("❌ Dataset not found.")
    print("\nSearching repository for CSV files:")

    for root, dirs, files in os.walk(repo_path):
        for file in files:
            if file.endswith(".csv"):
                print(os.path.join(root, file))

# Load dataset
df = pd.read_csv(data_path)

print("\n✅ Dataset loaded successfully!")
print("Shape:", df.shape)

Repository already exists.
✅ Dataset found!
Path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

✅ Dataset loaded successfully!
Shape: (30000, 44)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# ML-07 — Build the Ranked Queue

import numpy as np
import os

# Required columns
df["ctr"] = pd.to_numeric(df["ctr"], errors="coerce")
df["avg_position"] = pd.to_numeric(df["avg_position"], errors="coerce")

# 0 means no position data, so treat it as missing
valid_position = df["avg_position"].replace(0, np.nan)

# Thresholds
ctr_threshold = df["ctr"].median()
position_threshold = valid_position.median()

print("CTR threshold:", ctr_threshold)
print("Position threshold:", position_threshold)

# Baseline rule
df["baseline_reason_code"] = np.where(
    (df["ctr"] < ctr_threshold) &
    (valid_position > position_threshold),
    "CTR_LOW_POSITION",
    "MONITOR"
)

# Action
df["action"] = np.where(
    df["baseline_reason_code"] == "CTR_LOW_POSITION",
    "Prioritize Optimization",
    "Monitor"
)

# Score
ctr_gap = (
    (ctr_threshold - df["ctr"]) /
    (abs(ctr_threshold) + 1e-9)
)

position_gap = (
    (valid_position - position_threshold) /
    (abs(position_threshold) + 1e-9)
)

df["baseline_action_score"] = (
    ctr_gap.fillna(0) + position_gap.fillna(0)
)

# Only rule-matching pages get a priority score
df.loc[
    df["baseline_reason_code"] != "CTR_LOW_POSITION",
    "baseline_action_score"
] = 0

# Rank
df = df.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

df["baseline_rank"] = df.index + 1

# Create queue
output_columns = [
    "baseline_rank",
    "content_id",
    "ctr",
    "avg_position",
    "baseline_action_score",
    "baseline_reason_code",
    "action"
]

baseline_queue = df[output_columns].copy()

# Save output
output_dir = "/content/flyrank-ml-internship/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

baseline_queue.to_csv(output_path, index=False)

print("\n✅ Baseline ranked queue created!")
print("Rows:", len(baseline_queue))
print("Saved:", output_path)

display(baseline_queue.head(20))

CTR threshold: 0.07
Position threshold: 11.4

✅ Baseline ranked queue created!
Rows: 30000
Saved: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv


,baseline_rank,content_id,ctr,avg_position,baseline_action_score,baseline_reason_code,action
0,1,content_661e1745db72,0.0,245.0,21.491228,CTR_LOW_POSITION,Prioritize Optimization
1,2,content_23f1cc8851a9,0.0,184.0,16.140351,CTR_LOW_POSITION,Prioritize Optimization
2,3,content_7275a6a3a8eb,0.0,165.5,14.517544,CTR_LOW_POSITION,Prioritize Optimization
3,4,content_71a31b831092,0.0,161.0,14.122807,CTR_LOW_POSITION,Prioritize Optimization
4,5,content_42c7c72b8391,0.0,145.5,12.763158,CTR_LOW_POSITION,Prioritize Optimization
5,6,content_cb6c7d58c0bc,0.0,144.5,12.675439,CTR_LOW_POSITION,Prioritize Optimization
6,7,content_692fda8c52bd,0.0,142.0,12.456140,CTR_LOW_POSITION,Prioritize Optimization
7,8,content_3e087a5d8f15,0.0,138.8,12.175439,CTR_LOW_POSITION,Prioritize Optimization
8,9,content_13bbd72aea33,0.0,118.0,10.350877,CTR_LOW_POSITION,Prioritize Optimization
9,10,content_abeb1aa40158,0.0,113.5,9.956140,CTR_LOW_POSITION,Prioritize Optimization


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# ML-07 — Top-20 Review

top20 = baseline_queue.head(20).copy()

top20_review = top20.copy()

# Confidence note
top20_review["confidence_note"] = np.where(
    (top20_review["ctr"] < ctr_threshold) &
    (top20_review["avg_position"] > position_threshold),
    "Medium confidence: both low CTR and poor position are observed.",
    "Lower confidence: review the page signals before acting."
)

# What could make the recommendation wrong
top20_review["what_would_make_it_wrong"] = (
    "Search intent, seasonality, SERP features, or incomplete position data "
    "could explain the observed signal."
)

review_columns = [
    "baseline_rank",
    "content_id",
    "ctr",
    "avg_position",
    "baseline_action_score",
    "action",
    "baseline_reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20_review[review_columns]

display(top20_review)

,baseline_rank,content_id,ctr,avg_position,baseline_action_score,action,baseline_reason_code,confidence_note,what_would_make_it_wrong
0,1,content_661e1745db72,0.0,245.0,21.491228,Prioritize Optimization,CTR_LOW_POSITION,Medium confidence: both low CTR and poor posit...,"Search intent, seasonality, SERP features, or ..."
1,2,content_23f1cc8851a9,0.0,184.0,16.140351,Prioritize Optimization,CTR_LOW_POSITION,Medium confidence: both low CTR and poor posit...,"Search intent, seasonality, SERP features, or ..."
2,3,content_7275a6a3a8eb,0.0,165.5,14.517544,Prioritize Optimization,CTR_LOW_POSITION,Medium confidence: both low CTR and poor posit...,"Search intent, seasonality, SERP features, or ..."
3,4,content_71a31b831092,0.0,161.0,14.122807,Prioritize Optimization,CTR_LOW_POSITION,Medium confidence: both low CTR and poor posit...,"Search intent, seasonality, SERP features, or ..."
4,5,content_42c7c72b8391,0.0,145.5,12.763158,Prioritize Optimization,CTR_LOW_POSITION,Medium confidence: both low CTR and poor posit...,"Search intent, seasonality, SERP features, or ..."
5,6,content_cb6c7d58c0bc,0.0,144.5,12.675439,Prioritize Optimization,CTR_LOW_POSITION,Medium confidence: both low CTR and poor posit...,"Search intent, seasonality, SERP features, or ..."
6,7,content_692fda8c52bd,0.0,142.0,12.456140,Prioritize Optimization,CTR_LOW_POSITION,Medium confidence: both low CTR and poor posit...,"Search intent, seasonality, SERP features, or ..."
7,8,content_3e087a5d8f15,0.0,138.8,12.175439,Prioritize Optimization,CTR_LOW_POSITION,Medium confidence: both low CTR and poor posit...,"Search intent, seasonality, SERP features, or ..."
8,9,content_13bbd72aea33,0.0,118.0,10.350877,Prioritize Optimization,CTR_LOW_POSITION,Medium confidence: both low CTR and poor posit...,"Search intent, seasonality, SERP features, or ..."
9,10,content_abeb1aa40158,0.0,113.5,9.956140,Prioritize Optimization,CTR_LOW_POSITION,Medium confidence: both low CTR and poor posit...,"Search intent, seasonality, SERP features, or ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# ML-07 — Weak Picks + Leakage Check

# Weak picks: pages that do not meet the baseline rule
weak_picks = baseline_queue[
    baseline_queue["baseline_reason_code"] == "MONITOR"
].tail(10)

print("Weak / monitored picks:")
display(weak_picks)

# Leakage check
rule_features = {"ctr", "avg_position"}

print("\nLeakage Check")
print("Rule features used:", rule_features)
print("Target label used: No")
print("Future-window information used: No")
print("Client names / private queries used: No")

print("\n✅ Leakage check passed.")

Weak / monitored picks:


,baseline_rank,content_id,ctr,avg_position,baseline_action_score,baseline_reason_code,action
29990,29991,content_8bc4cd8e7ebc,0.16,7.0,0.0,MONITOR,Monitor
29991,29992,content_59369f4c2428,0.00,8.2,0.0,MONITOR,Monitor
29992,29993,content_e4750940e3ab,0.00,8.1,0.0,MONITOR,Monitor
29993,29994,content_a36795eb1410,0.00,1.0,0.0,MONITOR,Monitor
29994,29995,content_cacbfe2218b2,0.00,10.2,0.0,MONITOR,Monitor
29995,29996,content_ce78b1b0c99b,0.21,9.7,0.0,MONITOR,Monitor
29996,29997,content_353e8d87a41a,0.55,2.9,0.0,MONITOR,Monitor
29997,29998,content_8e1ae7310cd3,0.03,10.7,0.0,MONITOR,Monitor
29998,29999,content_b382fd6fa54f,0.16,12.2,0.0,MONITOR,Monitor
29999,30000,content_05867bb87732,0.31,21.1,0.0,MONITOR,Monitor



Leakage Check
Rule features used: {'avg_position', 'ctr'}
Target label used: No
Future-window information used: No
Client names / private queries used: No

✅ Leakage check passed.


In [12]:
import os

output_path = "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"

print("File exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    print("✅ baseline_action_score.csv successfully created!")
    print("File size:", os.path.getsize(output_path), "bytes")

File exists: True
✅ baseline_action_score.csv successfully created!
File size: 1981339 bytes


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.